# Exploration de Reviews — phase 1

Lecture seule des CSV sur C:, noyau .venv du projet. Exécuter de haut en bas depuis la racine ou notebooks/.
Grain à examiner : ni review_id ni order_id n'est supposé unique avant profilage.
Aucun commentaire intégral reproduit dans le rapport, aucun CSV réécrit et aucun validateur appelé.
Code exécuté avec Python le 2026-10-24 ; notebook livré sans sorties sauvegardées ni lancement Jupyter.
Rapport : docs/REVIEWS_EXPLORATION.md.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "AGENTS.md").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "AGENTS.md").is_file(), "Ouvrir depuis la racine ou notebooks/"
paths = {
    "reviews": PROJECT_ROOT / "data" / "olist_order_reviews_dataset.csv",
    "orders": PROJECT_ROOT / "data" / "olist_orders_dataset.csv",
}
hashes_before = {name: hashlib.sha256(path.read_bytes()).hexdigest()
                 for name, path in paths.items()}
reviews = pd.read_csv(paths["reviews"])
raw = pd.read_csv(paths["reviews"], dtype="string", keep_default_na=False)
orders = pd.read_csv(paths["orders"], usecols=[
    "order_id", "order_status", "order_purchase_timestamp",
    "order_delivered_customer_date",
])

def counts(series):
    return {str(key): int(value) for key, value in series.value_counts().items()}

def records(frame):
    return json.loads(frame.to_json(orient="records", date_format="iso"))

report = {
    "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                    "executable": sys.executable},
    "source_sha256": hashes_before,
    "source_size_bytes": paths["reviews"].stat().st_size,
    "shape": list(reviews.shape),
    "dtypes": {col: str(dtype) for col, dtype in reviews.dtypes.items()},
    "memory_deep_MiB": round(float(reviews.memory_usage(deep=True).sum()) / 2**20, 3),
    "missing": {col: int(value) for col, value in reviews.isna().sum().items()},
    "empty_strings": {col: int(raw[col].eq("").sum()) for col in raw},
    "whitespace_only_nonempty": {col: int((raw[col].ne("") & raw[col].str.strip().eq("")).sum())
                                 for col in raw},
    "outer_whitespace": {col: int(raw[col].ne(raw[col].str.strip()).sum()) for col in raw},
    "exact_duplicate_excess": int(reviews.duplicated().sum()),
    "exact_duplicate_involved_rows": int(reviews.duplicated(keep=False).sum()),
}
report

## Identifiants et cardinalités

Compter séparément les clés distinctes, les lignes impliquées dans des répétitions et les occurrences excédentaires.
Examiner la clé composite (review_id, order_id), les ID partagés entre commandes et les commandes ayant plusieurs avis.
Une unicité observée n'établit pas à elle seule une clé métier garantie par la source.

In [ ]:
id_profile = {}
for col in ["review_id", "order_id"]:
    frequency = reviews[col].value_counts()
    id_profile[col] = {
        "distinct": int(reviews[col].nunique()),
        "missing_or_blank_rows": int((reviews[col].isna() | raw[col].str.strip().eq("")).sum()),
        "unusual_hex_format_rows": int((~raw[col].str.fullmatch("[0-9a-f]{32}", na=False)).sum()),
        "repeated_values": int(frequency.gt(1).sum()),
        "rows_in_repeated_values": int(frequency[frequency.gt(1)].sum()),
        "excess_occurrences": int(reviews[col].duplicated().sum()),
        "frequency_distribution": counts(frequency),
        "max_occurrences": int(frequency.max()),
    }
review_orders = reviews.groupby("review_id").order_id.nunique()
review_id_repeated = reviews[reviews.review_id.duplicated(keep=False)]
payload_columns = [col for col in reviews.columns if col not in ["review_id", "order_id"]]
variation = review_id_repeated.groupby("review_id")[payload_columns].nunique(dropna=False)
report["identifiers"] = id_profile
report["composite_key"] = {
    "distinct_pairs": int(len(reviews[["review_id", "order_id"]].drop_duplicates())),
    "duplicate_excess": int(reviews.duplicated(["review_id", "order_id"]).sum()),
    "duplicate_involved_rows": int(reviews.duplicated(["review_id", "order_id"], keep=False).sum()),
    "review_ids_spanning_multiple_orders": int(review_orders.gt(1).sum()),
    "repeated_review_ids_with_identical_payload": int(variation.le(1).all(axis=1).sum()),
    "repeated_review_ids_with_varying_payload": int(variation.gt(1).any(axis=1).sum()),
}
report["identifiers"], report["composite_key"]
order_variation = reviews.groupby("order_id").agg(
    distinct_reviews=("review_id", "nunique"),
    distinct_scores=("review_score", "nunique"),
)
report["composite_key"]["orders_with_multiple_distinct_review_ids"] = int(
    order_variation.distinct_reviews.gt(1).sum())
report["composite_key"]["orders_with_varying_scores"] = int(
    order_variation.distinct_scores.gt(1).sum())
report["composite_key"]


## Notes et commentaires

Une note doit être convertible, finie, entière et comprise entre 1 et 5 selon le contrat proposé.
Un commentaire absent n'est pas une note invalide. Les espaces seuls sont considérés absents pour la complétude, en complément des chaînes vides.
Les statistiques de longueur ne prouvent pas la qualité du texte ; aucune correction automatique ni analyse de sentiment.

In [ ]:
score = pd.to_numeric(reviews.review_score, errors="coerce")
finite_score = score.notna() & np.isfinite(score)
report["scores"] = {
    "distribution": counts(reviews.review_score),
    "missing": int(reviews.review_score.isna().sum()),
    "non_numeric_present": int((reviews.review_score.notna() & score.isna()).sum()),
    "non_finite_present": int((score.notna() & ~np.isfinite(score)).sum()),
    "non_integer_finite": int((finite_score & score.where(finite_score).mod(1).ne(0)).sum()),
    "outside_1_5": int((finite_score & ~score.between(1, 5)).sum()),
    "mean": float(score.mean()),
}
comment_columns = ["review_comment_title", "review_comment_message"]
comment_missing = pd.DataFrame({col: raw[col].str.strip().eq("") for col in comment_columns})
report["comments"] = {
    col: {
        "missing_or_blank": int(comment_missing[col].sum()),
        "present": int((~comment_missing[col]).sum()),
        "length_statistics_present": raw.loc[~comment_missing[col], col].str.len().describe().to_dict(),
    } for col in comment_columns
}
report["comment_patterns"] = {
    "both_absent": int(comment_missing.all(axis=1).sum()),
    "any_absent_union": int(comment_missing.any(axis=1).sum()),
    "both_present": int((~comment_missing.any(axis=1)).sum()),
    "title_only": int((~comment_missing.review_comment_title & comment_missing.review_comment_message).sum()),
    "message_only": int((comment_missing.review_comment_title & ~comment_missing.review_comment_message).sum()),
}
report["scores"], report["comments"], report["comment_patterns"]

## Dates et délais

Conversion explicite du format observé %Y-%m-%d %H:%M:%S ; conversion mixed en diagnostic complémentaire.
Délais en jours de 24 h : review_answer_timestamp moins review_creation_date, sans interpréter cela comme un délai de réponse du vendeur.
Aucun fuseau horaire n'est fourni. Une date créée à minuit a une précision différente d'un horodatage de réponse.
Les repères de 7/30 jours sont descriptifs, sans seuil de rejet.

In [ ]:
dates = {}
date_profiles = {}
for col in ["review_creation_date", "review_answer_timestamp"]:
    values = raw[col]
    missing = values.str.strip().eq("")
    parsed = pd.to_datetime(values.where(~missing), format="%Y-%m-%d %H:%M:%S", errors="coerce")
    mixed = pd.to_datetime(values.where(~missing), format="mixed", errors="coerce")
    dates[col] = parsed
    date_profiles[col] = {
        "missing_or_blank": int(missing.sum()),
        "nonconvertible_present": int((~missing & parsed.isna()).sum()),
        "convertible_only_with_mixed": int((parsed.isna() & mixed.notna()).sum()),
        "format_mismatch_present": int((~missing & ~values.str.fullmatch(
            "[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2}", na=False)).sum()),
        "min": None if parsed.dropna().empty else str(parsed.min()),
        "max": None if parsed.dropna().empty else str(parsed.max()),
        "midnight_rows": int(parsed.eq(parsed.dt.normalize()).sum()),
    }
delay_days = (dates["review_answer_timestamp"] - dates["review_creation_date"]).dt.total_seconds() / 86400
report["dates"] = date_profiles
report["answer_delay_days"] = {
    "statistics": delay_days.describe(percentiles=[.25, .5, .75, .95, .99]).to_dict(),
    "negative_rows": int(delay_days.lt(0).sum()),
    "zero_rows": int(delay_days.eq(0).sum()),
    "over_7_days_rows": int(delay_days.gt(7).sum()),
    "over_30_days_rows": int(delay_days.gt(30).sum()),
}
report["dates"], report["answer_delay_days"]

## Relations avec Orders et chronologie

Jointure Reviews → Orders many_to_one, sans perte ou multiplication de lignes. Les commandes sans avis sont mesurées côté Orders.
Les avis multiples et la chronologie par rapport à la livraison sont des indicateurs métier à interpréter, pas des motifs de suppression.
Une jointure directe avec Items ou Payments pourrait multiplier les avis : définir le grain avant tout KPI.

In [ ]:
unknown = ~reviews.order_id.isin(orders.order_id)
no_review = ~orders.order_id.isin(reviews.order_id)
reviews_per_order = reviews.groupby("order_id").size()
joined = reviews.merge(orders, on="order_id", how="left", validate="many_to_one", indicator=True)
purchase = pd.to_datetime(joined.order_purchase_timestamp, format="%Y-%m-%d %H:%M:%S", errors="coerce")
delivery = pd.to_datetime(joined.order_delivered_customer_date, format="%Y-%m-%d %H:%M:%S", errors="coerce")
creation = pd.to_datetime(joined.review_creation_date, format="%Y-%m-%d %H:%M:%S", errors="coerce")
answer = pd.to_datetime(joined.review_answer_timestamp, format="%Y-%m-%d %H:%M:%S", errors="coerce")
creation_before_delivery = creation.lt(delivery)
report["relationships"] = {
    "orders_rows": len(orders),
    "unknown_order_review_rows": int(unknown.sum()),
    "unknown_distinct_order_ids": int(reviews.loc[unknown, "order_id"].nunique()),
    "orders_without_review": int(no_review.sum()),
    "orders_without_review_by_status": counts(orders.loc[no_review, "order_status"]),
    "orders_with_multiple_review_rows": int(reviews_per_order.gt(1).sum()),
    "review_rows_in_multi_review_orders": int(reviews_per_order[reviews_per_order.gt(1)].sum()),
    "reviews_per_order_distribution": counts(reviews_per_order),
    "joined_rows": len(joined),
    "review_rows_by_order_status": counts(joined.order_status),
}
report["order_chronology"] = {
    "creation_before_purchase_rows": int(creation.lt(purchase).sum()),
    "answer_before_purchase_rows": int(answer.lt(purchase).sum()),
    "rows_with_delivery_for_comparison": int(delivery.notna().sum()),
    "creation_before_delivery_rows": int(creation_before_delivery.sum()),
    "creation_before_delivery_calendar_day_rows": int(creation.dt.normalize().lt(delivery.dt.normalize()).sum()),
    "answer_before_delivery_rows": int(answer.lt(delivery).sum()),
    "creation_without_recorded_delivery_rows": int((creation.notna() & delivery.isna()).sum()),
}
assert len(joined) == len(reviews)
assert hashes_before == {name: hashlib.sha256(path.read_bytes()).hexdigest()
                         for name, path in paths.items()}
report["relationships"], report["order_chronology"]

## Proposition de contrat — en attente de validation

Sept colonnes obligatoires, IDs renseignés, note finie entière de 1 à 5, dates renseignées et convertibles, référence order_id existante.
Examiner les répétitions avant d'imposer une clé. La paire (review_id, order_id) est une candidate seulement si le profilage confirme son unicité ; pas de règle unique(review_id) ou unique(order_id) par intuition.
Commentaires facultatifs ; absences, avis multiples, répétitions d'ID et chronologie suivis séparément.
Sans stratégie de sélection validée, conserver toutes les lignes et ne pas choisir arbitrairement le premier/dernier avis.
Synthèse chiffrée ci-dessous, sans statut valid/review ni sortie de pipeline.

In [ ]:
print(json.dumps(report, ensure_ascii=True, indent=2))